In [ ]:
from retrieval.index import DistributedIndex, load_or_initialize_index, build_index
import yaml
import os
from pathlib import Path
import torch
import json
import ir_datasets
import numpy as np
import mteb

In [ ]:
def load_model_meta_yaml(file_path : str | Path) -> dict:
    with open(file_path, "r") as f:
        return yaml.safe_load(f)

In [ ]:
path = Path("/home/rjha5/603-nvme2/arena/model_meta.yml")

In [ ]:
model_meta = load_model_meta_yaml(path)


In [ ]:
[model for model in model_meta["model_meta"].keys() if model_meta["model_meta"][model].get("size", 7000) < 2000]

In [ ]:
from models import ModelManager


model_manager = ModelManager(model_meta=model_meta)

In [ ]:
model = model_manager.load_model("intfloat/multilingual-e5-small")

In [ ]:
model_manager.load_local_index(model_name="intfloat/multilingual-e5-small", corpus="wikipedia", embedbs=1024)

# FOOBAR

In [ ]:
# write dummy passages to a jsonl file
with open("dummy_passages.jsonl", "w") as f:
    for i in range(100):
        f.write(json.dumps({"_id": i, "title": f"Dummy Passage {i}", "text": f"This is a dummy passage {i}"}) + "\n")

In [ ]:
model_name = 'sentence-transformers/all-MiniLM-L6-v2'
# model_name = 'intfloat/multilingual-e5-small'
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = mteb.get_model(model_name, revision=model_meta["model_meta"][model_name].get('revision', None), device=device)


In [ ]:
index, passages = load_or_initialize_index(dim=384, passages=["dummy_passages.jsonl"])

In [ ]:
build_index(model.bfloat16(), index, [p["text"] for p in passages], gpu_embedder_batch_size=256)

In [ ]:
index.embeddings

In [ ]:
emb_normed = torch.nn.functional.normalize(index.embeddings, p=2, dim=1)
emb_normed.norm(dim=1)

# Model Manager

In [ ]:
model_meta

In [ ]:
model_name = "nomic-ai/nomic-embed-text-v1.5"
model_name = "BAAI/bge-large-en-v1.5"
model = mteb.get_model(model_name, revision=model_meta["model_meta"][model_name].get('revision', None), device=device)

In [ ]:
nomic = mteb.get_model("nomic-ai/nomic-embed-text-v1.5")
nomic

In [ ]:
hasattr(nomic, "encode_corpus")

In [ ]:
x = nomic.encode_corpus(["foobar", "baz"], convert_to_tensor=False)
type(x), x

In [ ]:
y = nomic.encode_corpus(["foobar", "baz"], convert_to_tensor=True)
type(y), y

In [ ]:
z = nomic.encode_corpus(["foobar", "baz"])
type(z), z

In [ ]:
model_meta["model_meta"][model_name]


In [ ]:
from mteb import Encoder
from typing import Any

from retrieval.index import DTYPE_TO_TORCH_DTYPE

def index_collection(model : Encoder, collection : list[str], model_meta : dict[str, Any] = {}, batch_size=32) -> DistributedIndex:
    
    index = DistributedIndex(dtype=DTYPE_TO_TORCH_DTYPE[model_meta.get("index_dtype", "float32")])
    index.init_embeddings(collection, dim=model_meta["dim"])

    print(index.embeddings.dtype)

    build_index(model, index, collection, gpu_embedder_batch_size=batch_size)

    return index

In [ ]:
index = index_collection(model, collection=[f"foobar the {i}th was a mighty king" for i in range(1000)], model_meta=model_meta["model_meta"][model_name], batch_size=64)

In [ ]:
index.search_knn(model.encode(["foobar doc 1", "foobar doc 42"], convert_to_tensor=True), topk=5)

In [ ]:
import datasets
from tqdm import tqdm

In [ ]:
wikipedia = datasets.load_dataset("orionweller/wikipedia-2024-06-24-docs", split="train")

In [ ]:
wikipedia

In [ ]:
title_text = [f"{title}\n{text}" for title, text in tqdm(zip(wikipedia["title"], wikipedia["text"]), total=len(wikipedia))]

In [ ]:
index = index_collection()

In [ ]:
from retrieval.common import load_passages_from_hf

In [ ]:
wiki = load_passages_from_hf("wikipedia", limit=None)

In [ ]:
wiki

# Simple indexing

In [ ]:
from mteb import get_model
from datasets import load_dataset
import numpy as np
from tqdm.auto import tqdm
import time
import torch

In [ ]:
wiki = load_dataset("mteb/arena-wikipedia-7-15-24", split="train")

In [ ]:
# model = get_model("jinaai/jina-embeddings-v2-base-en", revision="31b72fbf354fea65264ec54edf0b189d94b92d39")
model = get_model("BAAI/bge-large-en-v1.5", revision="d4aa6901d3a41ba39fb536a557fa166f842b0e09")
# model = get_model("mixedbread-ai/mxbai-embed-large-v1", revision="990580e27d329c7408b3741ecff85876e128e203")
# model = get_model("nomic-ai/nomic-embed-text-v1.5", revision="b0753ae76394dd36bcfb912a46018088bca48be0")

In [ ]:
start = time.time()
x = model.encode(wiki["text"][:5_000], convert_to_tensor=True, batch_size=1600, show_progress_bar=True)
end = time.time()
print(f"Time taken: {end - start} seconds")

# mem usage:

In [ ]:
batch, dim = 1024, 32

x , y = torch.zeros(dim, batch), torch.zeros(dim, batch)
z = torch.cat([x, y], dim=1)
z.shape

In [ ]:
if not isinstance(x, torch.Tensor):
    x = torch.tensor(x)

In [ ]:
gp_per_x = (x.nelement() * x.element_size() / 1024 ** 3)

gb_per_4M = (4_000_000 / x.shape[0]) * gp_per_x

hrs_per_x = (end - start) / 3600

hrs_per_4M = hrs_per_x * (4_000_000 / x.shape[0])

print(f"{gb_per_4M=:.2f}GB, {hrs_per_4M=:.2f}hrs to embed 4M docs")

# BGE large: 15 GB, 21.1 hrs, ~50% vram usage (bs=1024)
# MBAI large: 15 GB, 21 hrs, ~50% vram usage (bs=1024)
# Nomic embed v1.5: 12 GB, 10.1 hrs, ~80% vram usage (bs=1024)

In [ ]:
x = torch.load("/home/hltcoe/rjha/arena/index_wikipedia_BAAI_bge-large-en-v1.5/embeddings.0.pt")

In [ ]:
x.shape